# 4 · Classificador complementar por segmentação de faixas

O segmentador HiRes já estudado no projeto identifica cores por pixel; a decodificação usa o eixo e a ordem das regiões. Este é um teste complementar em Eralp, separado do Barrettotte usado em dados do HiRes. O histórico completo de treinamento do checkpoint de terceiros não foi auditado; por isso, este experimento **não comprova independência entre os dados de treino e teste**. Não usar como resultado final de generalização.

O detector de objetos vem do notebook 1. As imagens Eralp têm rótulo nominal, mas as caixas do manifesto abrangem a imagem inteira e a tolerância está ausente. O teste inclui os casos sem detecção ou sem leitura no denominador.

In [ ]:
from pathlib import Path
import os,sys,json,hashlib,time
from collections import Counter
import cv2,numpy as np,pandas as pd,torch
from ultralytics import YOLO
def read_image(path):
    try: return cv2.imdecode(np.fromfile(str(path),dtype=np.uint8),cv2.IMREAD_COLOR)
    except (OSError,cv2.error): return None

PROJECT=Path(os.environ.get("RESISTOR_PROJECT_ROOT","..")).expanduser().resolve()
DATA=Path(os.environ.get("RESISTOR_DATA_ROOT",str(PROJECT/"data"))).expanduser().resolve()
OUTPUT=Path(os.environ.get("RESISTOR_OUTPUT",str(PROJECT/"outputs/proposta"))).expanduser().resolve()
HIRES=DATA/"sources/github-hires"
WEIGHTS=Path(os.environ.get("RESISTOR_YOLO_WEIGHTS",str(OUTPUT/"treino/yolo-samu/weights/best.pt"))).expanduser()
SEG_WEIGHTS=Path(os.environ.get("RESISTOR_HIRES_WEIGHTS",str(HIRES/"weights/segmentation/efficientnet-b2_best.pt"))).expanduser()
for p in (WEIGHTS,SEG_WEIGHTS,HIRES/"segmentation",HIRES/"band_extraction"):
    assert p.exists(), f"Dependência não encontrada: {p}"
sys.path.insert(0,str(HIRES/"segmentation"));sys.path.insert(0,str(HIRES/"band_extraction"))
from infer import load_model, _run_pass
from dataset import get_val_transform, IMG_SIZE
from resistance_calculator import calculate_resistance_with_axis_info
from color_code_tables import ResistanceResult
sys.path.insert(0,str(PROJECT/'src'))
from resistor_detector.matlab_compat import analyze_matlab_compatible
DEVICE=torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
SEG_SIZE=int(os.environ.get("RESISTOR_SEG_SIZE",str(IMG_SIZE)))
detector=YOLO(str(WEIGHTS)); segmenter=load_model(str(SEG_WEIGHTS),DEVICE); transform=get_val_transform()
OUTPUT.mkdir(parents=True,exist_ok=True)
threshold_path=OUTPUT/"limiar_escolhido_validacao.json"
threshold=float(os.environ.get("RESISTOR_CONFIDENCE",json.loads(threshold_path.read_text())["limiar"] if threshold_path.exists() else .25))
print("Detector:",WEIGHTS,"Segmentador:",SEG_WEIGHTS,"Dispositivo:",DEVICE)


## Inferência em memória

A máscara é calculada em duas passagens como na referência HiRes; a segunda usa a região não vazia da primeira com margem. As imagens de entrada não são gravadas pelo notebook. A confiança da segmentação é uma pontuação do modelo, não confiança calibrada do valor.

In [ ]:
def hires_read(crop):
    if crop is None or min(crop.shape[:2])<20:
        return {"ohms":None,"tolerancia":None,"status":"recorte insuficiente","segmentacao":None}
    rgb=cv2.cvtColor(crop,cv2.COLOR_BGR2RGB)
    mask,seg_conf=_run_pass(segmenter,rgb,transform,DEVICE,SEG_SIZE)
    rows=np.any(mask>0,axis=1); cols=np.any(mask>0,axis=0)
    if rows.any() and cols.any():
        rmin,rmax=int(np.where(rows)[0][0]),int(np.where(rows)[0][-1])+1
        cmin,cmax=int(np.where(cols)[0][0]),int(np.where(cols)[0][-1])+1
        py=max(8,int((rmax-rmin)*.2)); px=max(8,int((cmax-cmin)*.2))
        r0,r1=max(0,rmin-py),min(rgb.shape[0],rmax+py)
        c0,c1=max(0,cmin-px),min(rgb.shape[1],cmax+px)
        if min(r1-r0,c1-c0)>=20:
            part,seg_conf=_run_pass(segmenter,rgb[r0:r1,c0:c1],transform,DEVICE,SEG_SIZE)
            mask=np.zeros(rgb.shape[:2],dtype=np.uint8); mask[r0:r1,c0:c1]=part
    result,axis=calculate_resistance_with_axis_info(mask)
    if not isinstance(result,ResistanceResult):
        return {"ohms":None,"tolerancia":None,"status":"revisar","segmentacao":float(seg_conf)}
    return {"ohms":float(result.value),"tolerancia":float(result.tolerance),"status":"candidato: revisar",
            "faixas":[str(b.color_name) for b in result.bands],"segmentacao":float(seg_conf)}


## Avaliação externa exploratória

As pastas de valor são agrupadas deterministicamente antes da seleção de teste; a identidade física das peças não é conhecida. A amostra padrão contém até 200 imagens. O detector seleciona a caixa de maior confiança; não há caixas confiáveis para verificar se ela pertence realmente ao resistor. `tolerância` não é avaliada porque o Eralp não a rotula.

In [ ]:
manifest=PROJECT/"data/manifests/eralpozcan-crop-value.jsonl"
rows=[json.loads(s) for s in manifest.read_text(encoding="utf-8").splitlines() if s.strip()]
for row in rows:
    rel=Path(row["path"]);row["_path"]=DATA/rel.relative_to("data");row["_group"]=rel.parent.name
groups=sorted({r["_group"] for r in rows},key=lambda g:hashlib.sha256(("42:"+g).encode()).hexdigest())
selected=set(groups[int(.85*len(groups)):])
test=[r for r in rows if r["_group"] in selected]
random_seed=np.random.default_rng(42); limit=int(os.environ.get("RESISTOR_HIRES_SAMPLE","200"))
if len(test)>limit: test=[test[i] for i in sorted(random_seed.choice(len(test),limit,replace=False))]
assert test and all(r["_path"].exists() for r in test)
print("Grupos de teste:",len(selected),"imagens:",len(test))


In [ ]:
records=[]
for row in test:
    frame=read_image(row["_path"]); assert frame is not None
    start=time.perf_counter()
    detections=detector.predict(frame,conf=threshold,iou=.7,device=0 if torch.cuda.is_available() else "cpu",verbose=False)[0].boxes
    if len(detections):
        i=int(detections.conf.argmax()); x1,y1,x2,y2=map(int,detections.xyxy[i].cpu().numpy())
        h,w=frame.shape[:2]; pad_x=int((x2-x1)*.1);pad_y=int((y2-y1)*.1)
        crop=frame[max(0,y1-pad_y):min(h,y2+pad_y),max(0,x1-pad_x):min(w,x2+pad_x)]
        result=hires_read(crop)
    else: result={"ohms":None,"tolerancia":None,"status":"sem detecção","segmentacao":None}
    actual=float(row["objects"][0]["ohms"]); predicted=result["ohms"]
    records.append({"id":row["id"],"real_ohms":actual,"predito_ohms":predicted,"correto":predicted is not None and np.isclose(predicted,actual,rtol=1e-9),
                    "status":result["status"],"deteccoes":len(detections),"segmentacao":result["segmentacao"],"ms":1000*(time.perf_counter()-start)})
results=pd.DataFrame(records);results.to_csv(OUTPUT/"hires_eralp_por_imagem.csv",index=False)
emitted=results.predito_ohms.notna()
summary={"imagens":len(results),"cobertura":float(emitted.mean()),"acerto_total":float(results.correto.mean()),
         "acerto_emitidos":float(results.loc[emitted,"correto"].mean()) if emitted.any() else None,
         "tempo_total_medio_ms":float(results.ms.mean()),"ressalva":"independência do checkpoint de terceiros não verificada"}
(OUTPUT/"hires_eralp_resumo.json").write_text(json.dumps(summary,indent=2,ensure_ascii=False),encoding="utf-8")
display(pd.DataFrame([summary]))


A métrica mede um fluxo em imagens com um único resistor; não valida contagem em cenas com vários objetos. Ela não substitui teste próprio com caixas e valores anotados independentemente.

## Imagem própria com vários resistores
Defina `RESISTOR_IMAGE` para executar a detecção e a leitura individual. A leitura HiRes é complementada pelo método compatível com MATLAB quando as caixas coincidem; divergências pedem revisão. O método MATLAB não identifica tolerância, que permanece ausente. `RESISTOR_IMGSZ` permite mudar a resolução (padrão 640); nesse caso o limiar escolhido no notebook 5 precisa ser validado novamente. Valores emitidos são candidatos não calibrados.

In [ ]:
import matplotlib.pyplot as plt
image_path=os.environ.get("RESISTOR_IMAGE")
if image_path:
    frame=read_image(image_path)
    if frame is None: raise FileNotFoundError(image_path)
    started=time.perf_counter()
    image_size=int(os.environ.get("RESISTOR_IMGSZ","640"))
    predictions=detector.predict(frame,imgsz=image_size,conf=threshold,iou=.7,device=0 if torch.cuda.is_available() else "cpu",verbose=False)[0].boxes
    items=sorted(zip(predictions.xyxy.cpu().numpy(),predictions.conf.cpu().numpy()),key=lambda item:(item[0][1],item[0][0]))
    matlab=analyze_matlab_compatible(frame)["detections"]
    def box_iou(a,b):
        x=max(0,min(a[2],b[2])-max(a[0],b[0]));y=max(0,min(a[3],b[3])-max(a[1],b[1]));inter=x*y
        return inter/max(1e-9,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter)
    matlab_boxes=[[x,y,x+bw,y+bh] for x,y,bw,bh in (m["bbox"] for m in matlab)]
    assignments={};used_detector=set();used_matlab=set()
    pairs=sorted(((box_iou(box,mb),i,j) for i,(box,_) in enumerate(items) for j,mb in enumerate(matlab_boxes)),reverse=True)
    for score,i,j in pairs:
        if score<.5: break
        if i not in used_detector and j not in used_matlab:assignments[i]=j;used_detector.add(i);used_matlab.add(j)
    results=[]; illustrated=frame.copy(); h,w=frame.shape[:2]
    for number,(box,score) in enumerate(items,1):
        x1,y1,x2,y2=map(int,box); x1,y1=max(0,x1),max(0,y1); x2,y2=min(w,x2),min(h,y2)
        dx,dy=int((x2-x1)*.1),int((y2-y1)*.1)
        crop=frame[max(0,y1-dy):min(h,y2+dy),max(0,x1-dx):min(w,x2+dx)]
        reading=hires_read(crop)
        reference=matlab[assignments[number-1]] if number-1 in assignments else None
        fallback=reference.get("value_ohms") if reference else None
        if fallback is not None and reading["ohms"] is None:
            reading.update({"ohms":float(fallback),"tolerancia":None,"faixas":reference["bands"],"status":"candidato MATLAB: revisar tolerância"})
        elif fallback is not None and not np.isclose(fallback,reading["ohms"]):
            reading.update({"alternativas_ohms":[reading["ohms"],float(fallback)],"ohms":None,"tolerancia":None,"status":"revisar: métodos discordam"})
        results.append({"resistor":f"R{number}","caixa":[x1,y1,x2,y2],"confianca_deteccao":float(score),**reading})
        cv2.rectangle(illustrated,(x1,y1),(x2,y2),(40,220,80),2)
        cv2.putText(illustrated,f"R{number}",(x1,max(15,y1-5)),cv2.FONT_HERSHEY_SIMPLEX,.6,(40,220,80),2)
    latency_ms=1000*(time.perf_counter()-started)
    print("Resistores detectados:",len(results),"· tempo total (ms):",round(latency_ms,1)); display(pd.DataFrame(results))
    plt.figure(figsize=(12,8));plt.imshow(cv2.cvtColor(illustrated,cv2.COLOR_BGR2RGB));plt.axis("off");plt.show()
    prediction_path=os.environ.get("RESISTOR_OWN_PREDICTIONS"); image_id=os.environ.get("RESISTOR_IMAGE_ID")
    if prediction_path and image_id:
        out=Path(prediction_path).expanduser();out.parent.mkdir(parents=True,exist_ok=True)
        existing=[json.loads(s) for s in out.read_text(encoding="utf-8").splitlines() if s.strip()] if out.exists() else []
        if any(row.get("id")==image_id for row in existing): raise ValueError("Já existe previsão para este id; use outro arquivo ou remova a linha explicitamente.")
        exported={"id":image_id,"latency_ms":latency_ms,"objects":[{"bbox":r["caixa"],"ohms":r.get("ohms"),"tolerance_percent":r.get("tolerancia")} for r in results]}
        with out.open("a",encoding="utf-8") as handle:handle.write(json.dumps(exported,ensure_ascii=False)+"\n")
        print("Metadados de previsão:",out)
else: print("Defina RESISTOR_IMAGE para testar uma imagem.")


## Comparação opcional com referência individual
Defina `RESISTOR_REFERENCE` para um JSON de caixas e valores da mesma imagem. A associação usa IoU ≥ 0,5. Uma imagem com quatro peças serve apenas como teste de funcionamento; não estima desempenho geral.

In [ ]:
reference_path=os.environ.get("RESISTOR_REFERENCE")
if reference_path and image_path:
    reference=json.loads(Path(reference_path).read_text(encoding="utf-8"))
    if reference.get("image_size") != [frame.shape[1],frame.shape[0]]: raise ValueError("Referência e imagem têm dimensões diferentes")
    def iou(a,b):
        x=max(0,min(a[2],b[2])-max(a[0],b[0])); y=max(0,min(a[3],b[3])-max(a[1],b[1])); inter=x*y
        return inter/max(1e-9,(a[2]-a[0])*(a[3]-a[1])+(b[2]-b[0])*(b[3]-b[1])-inter)
    truth=reference["objects"];pairs=sorted(((iou(a["bbox"],b["caixa"]),i,j) for i,a in enumerate(truth) for j,b in enumerate(results)),reverse=True)
    used_truth,used_pred=set(),set(); matches={}
    for score,i,j in pairs:
        if score<.5:break
        if i not in used_truth and j not in used_pred:used_truth.add(i);used_pred.add(j);matches[i]=(j,score)
    comparison=[]
    for i,item in enumerate(truth):
        match=matches.get(i); pred=results[match[0]] if match else {}
        value=pred.get("ohms");tol=pred.get("tolerancia")
        comparison.append({"referencia":item["id"],"detectado":pred.get("resistor"),"iou":match[1] if match else None,
            "valor_real":item["ohms"],"valor_previsto":value,"valor_correto":value is not None and np.isclose(value,item["ohms"]),
            "tolerancia_real":item["tolerance_percent"],"tolerancia_prevista":tol,"status":pred.get("status","não detectado")})
    table=pd.DataFrame(comparison);display(table)
    print(f"Referência: {len(truth)} · detectados: {len(results)} · associados: {len(matches)} · valores corretos: {table.valor_correto.sum()}")
    table.to_csv(OUTPUT/"comparacao_imagem_referencia.csv",index=False)
elif reference_path: print("Informe também RESISTOR_IMAGE.")
